# QueryMind — Natural Language to SQL (OpenAI + MySQL)

## Project Overview

QueryMind takes a natural-language question, uses an OpenAI chat model to translate it into a
SQL query against a MySQL database, executes that query, and turns the raw result back into a
plain-language answer.

**Pipeline:** Question &rarr; OpenAI generates SQL &rarr; SQL runs on MySQL &rarr; OpenAI turns the result into a final answer.

- **LLM:** OpenAI `gpt-4.1-mini` via the `langchain-openai` chat model
- **Database:** MySQL (`text_to_sql`), accessed through LangChain's `SQLDatabase` utility
- **Orchestration:** LangChain Expression Language (LCEL) — two small chains, no agents, no RAG, no vector store
- **Evaluation:** [RAGAS](https://docs.ragas.io/) scores the generated SQL against a small reference set, using the same OpenAI model as judge


## Imports

In [1]:
import os
import re
import warnings
from getpass import getpass
from urllib.parse import quote_plus

warnings.filterwarnings("ignore", category=DeprecationWarning)

from langchain_community.utilities import SQLDatabase
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

## OpenAI Configuration

The API key is read from the `OPENAI_API_KEY` environment variable. If it isn't set, you'll be
prompted for it (via `getpass`, so it isn't echoed to the notebook output). The key is never
hardcoded or printed.

In [2]:
openai_api_key = os.environ.get("OPENAI_API_KEY") or getpass("Enter your OPENAI_API_KEY: ")

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    api_key=openai_api_key,
    temperature=0,
)

## MySQL Database Connection

Connects to the local MySQL `text_to_sql` database. The password is read from the
`MYSQL_PASSWORD` environment variable (or prompted for) and URL-encoded before being placed in
the connection URI, so special characters such as `@` don't break the connection string.

In [3]:
host = "localhost"
port = "3306"
username = "root"
database = "text_to_sql"

mysql_password = os.environ.get("MYSQL_PASSWORD") or getpass("Enter your MySQL password: ")

mysql_uri = f"mysql+pymysql://{username}:{quote_plus(mysql_password)}@{host}:{port}/{database}"
db = SQLDatabase.from_uri(mysql_uri, sample_rows_in_table_info=2)

print("Connected to:", db.dialect, "-", database)
print("Tables:", db.get_usable_table_names())

Connected to: mysql - text_to_sql
Tables: ['2017_budgets', 'customers', 'products', 'regions', 'sales_order', 'state_regions']


## Database Schema

`SQLDatabase.get_table_info()` returns the `CREATE TABLE` statements plus a few sample rows for
each table. This is fed to the LLM as context so it can write queries against the real schema
(table names, column names, and types) instead of guessing.

In [4]:
schema = db.get_table_info()
print(schema)


CREATE TABLE `2017_budgets` (
	`Product Name` TEXT, 
	`2017 Budgets` DOUBLE
)DEFAULT CHARSET=utf8mb4 ENGINE=InnoDB COLLATE utf8mb4_0900_ai_ci

/*
2 rows from 2017_budgets table:
Product Name	2017 Budgets
Product 1	3016489.2089999998
Product 2	3050087.5649999999
*/


CREATE TABLE customers (
	`Customer Index` INTEGER, 
	`Customer Names` TEXT
)DEFAULT CHARSET=utf8mb4 ENGINE=InnoDB COLLATE utf8mb4_0900_ai_ci

/*
2 rows from customers table:
Customer Index	Customer Names
1	Geiss Company
2	Jaxbean Group
*/


CREATE TABLE products (
	`Index` INTEGER, 
	`Product Name` TEXT
)DEFAULT CHARSET=utf8mb4 ENGINE=InnoDB COLLATE utf8mb4_0900_ai_ci

/*
2 rows from products table:
Index	Product Name
1	Product 1
2	Product 2
*/


CREATE TABLE regions (
	id INTEGER, 
	name TEXT, 
	county TEXT, 
	state_code TEXT, 
	state TEXT, 
	type TEXT, 
	latitude DOUBLE, 
	longitude DOUBLE, 
	area_code INTEGER, 
	population INTEGER, 
	households INTEGER, 
	median_income INTEGER, 
	land_area INTEGER, 
	water_area INTEGER

## SQL Generation Chain

An LCEL chain: `RunnablePassthrough.assign(schema=...)` attaches the live schema to the input,
a prompt asks the model for a single-line MySQL query, and `StrOutputParser` extracts the text.
`clean_sql` strips any Markdown code fences and a trailing semicolon that the model might add.

In [5]:
sql_gen_template = """You are a MySQL expert. Given an input question, write a syntactically \
correct MySQL query to answer it.
Only use the tables and columns shown in the schema below. Return ONLY the SQL query — no \
explanation, no Markdown code fences, no trailing semicolon.

Schema:
{schema}

Question: {question}
SQL Query:"""

sql_gen_prompt = ChatPromptTemplate.from_template(sql_gen_template)


def get_schema(_):
    return db.get_table_info()


def clean_sql(text: str) -> str:
    text = text.strip()
    match = re.search(r"```sql\s*(.*?)\s*```", text, re.DOTALL | re.IGNORECASE)
    if match:
        text = match.group(1).strip()
    return text.rstrip(";").strip()


sql_generation_chain = (
    RunnablePassthrough.assign(schema=get_schema)
    | sql_gen_prompt
    | llm
    | StrOutputParser()
    | clean_sql
)

## SQL Execution

`db.run(...)` executes the generated SQL against MySQL and returns the result as a string of
Python-literal rows.

In [6]:
def ask_database(question: str):
    sql_query = sql_generation_chain.invoke({"question": question})
    result = db.run(sql_query)
    return sql_query, result

## Natural Language Question

In [7]:
question = "What was the 2017 budget for Product 12?"

sql_query, result = ask_database(question)
print("Generated SQL:", sql_query)
print("Result:", result)

Generated SQL: SELECT `2017 Budgets` FROM `2017_budgets` WHERE `Product Name` = 'Product 12'
Result: [(1356976.996,)]


## Final Answer

A second, small LCEL chain turns the question, the SQL query, and the raw SQL result into a
concise natural-language answer.

In [8]:
answer_template = """Given the question, the SQL query used, and the SQL result, write a \
concise natural-language answer.

Question: {question}
SQL Query: {query}
SQL Result: {result}

Answer:"""

answer_prompt = ChatPromptTemplate.from_template(answer_template)
answer_chain = answer_prompt | llm | StrOutputParser()

final_answer = answer_chain.invoke(
    {"question": question, "query": sql_query, "result": result}
)
print(final_answer)

The 2017 budget for Product 12 was $1,356,976.996.


## RAGAS Evaluation

[RAGAS](https://docs.ragas.io/) scores the SQL-generation chain automatically, using an LLM as
judge instead of hand-checking each answer. Reusing the same `llm` (`gpt-4.1-mini`) that
generates the SQL keeps this to a single OpenAI setup — no extra provider, no vector store.

Two metrics, matching what the project's evaluation actually measures:
- **Context Precision** — whether the database schema given to the model is actually relevant to producing the expected SQL query.
- **Helpfulness** (a `RubricsScore`) — a 1-5 rubric judging how well the generated SQL satisfies the question.

In [9]:
from ragas.llms import LangchainLLMWrapper
from ragas import evaluate
from ragas.metrics import ContextPrecision, RubricsScore
from ragas.dataset_schema import SingleTurnSample, EvaluationDataset

evaluator_llm = LangchainLLMWrapper(llm)

helpfulness_rubrics = {
    "score1_description": "Response is useless/irrelevant, contains inaccurate/deceptive/misleading information, and/or contains harmful/offensive content. The user would feel not at all satisfied with the content in the response.",
    "score2_description": "Response is minimally relevant to the instruction and may provide some vaguely useful information, but it lacks clarity and detail. It might contain minor inaccuracies. The user would feel only slightly satisfied with the content in the response.",
    "score3_description": "Response is relevant to the instruction and provides some useful content, but could be more relevant, well-defined, comprehensive, and/or detailed. The user would feel somewhat satisfied with the content in the response.",
    "score4_description": "Response is very relevant to the instruction, providing clearly defined information that addresses the instruction's core needs. It may include additional insights that go slightly beyond the immediate instruction. The user would feel quite satisfied with the content in the response.",
    "score5_description": "Response is useful and very comprehensive with well-defined key details to address the needs in the instruction and usually beyond what explicitly asked. The user would feel very satisfied with the content in the response.",
}

context_precision = ContextPrecision(llm=evaluator_llm)
rubrics_score = RubricsScore(name="helpfulness", rubrics=helpfulness_rubrics, llm=evaluator_llm)

C:\Users\BIT\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


A small held-out set of questions with their expected SQL, run through the same `sql_generation_chain` used above.

In [10]:
eval_questions = [
    "What was the 2017 budget for Product 12?",
    "What are the names of all products in the products table?",
    "List all customer names from the customers table.",
    "Find the name and state of all regions in the regions table.",
    "What is the name of the customer with Customer Index = 1?",
]

eval_references = [
    "SELECT `2017 Budgets` FROM `2017_budgets` WHERE `Product Name` = 'Product 12';",
    "SELECT `Product Name` FROM products;",
    "SELECT `Customer Names` FROM customers;",
    "SELECT name, state FROM regions;",
    "SELECT `Customer Names` FROM customers WHERE `Customer Index` = 1;",
]

samples = []
for eval_question, eval_reference in zip(eval_questions, eval_references):
    generated_sql = sql_generation_chain.invoke({"question": eval_question})
    samples.append(
        SingleTurnSample(
            user_input=eval_question,
            retrieved_contexts=[schema],
            response=generated_sql,
            reference=eval_reference,
        )
    )

ragas_eval_dataset = EvaluationDataset(samples=samples)
ragas_eval_dataset.to_pandas()

,user_input,retrieved_contexts,response,reference
0,What was the 2017 budget for Product 12?,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,SELECT `2017 Budgets` FROM `2017_budgets` WHER...,SELECT `2017 Budgets` FROM `2017_budgets` WHER...
1,What are the names of all products in the prod...,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,SELECT `Product Name` FROM products,SELECT `Product Name` FROM products;
2,List all customer names from the customers table.,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,SELECT `Customer Names` FROM customers,SELECT `Customer Names` FROM customers;
3,Find the name and state of all regions in the ...,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,"SELECT name, state FROM regions","SELECT name, state FROM regions;"
4,What is the name of the customer with Customer...,[\nCREATE TABLE `2017_budgets` (\n\t`Product N...,SELECT `Customer Names` FROM customers WHERE `...,SELECT `Customer Names` FROM customers WHERE `...


In [11]:
ragas_result = evaluate(dataset=ragas_eval_dataset, metrics=[context_precision, rubrics_score])
ragas_result

Evaluating:   0%|          | 0/10 [00:00<?, ?it/s]

Evaluating:  10%|█         | 1/10 [00:24<03:40, 24.47s/it]

Evaluating: 100%|██████████| 10/10 [00:24<00:00,  2.45s/it]

{'context_precision': 1.0000, 'helpfulness': 4.6000}